# MarketPulse AI - End-to-End Workflow

This notebook orchestrates the existing Python pipeline.

**Full mode:** collect market data, engineer features, train models, backtest strategies, analyze performance, and generate latest-date predictions.

**Cached mode:** reuse locally downloaded market data and execute the remaining processing stages.

The notebook does not place real trades.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / 'src/run_pipeline.py').exists()
print('Project root:', PROJECT_ROOT)

## 1. Select execution mode

Set `MODE` to `'cached'` or `'full'`.
Set `DRY_RUN` to `True` to preview the workflow without executing it.

In [ ]:
MODE = 'cached'
DRY_RUN = True

assert MODE in {'cached', 'full'}
print('Mode:', MODE)
print('Dry run:', DRY_RUN)

## 2. Execute the Python pipeline

The orchestrator calls the existing scripts in `src/pipeline/` in dependency order.

Set `DRY_RUN = False` above to execute the workflow.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'src/run_pipeline.py'),
    '--mode', MODE,
]

if DRY_RUN:
    command.append('--dry-run')

result = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    check=True,
)

print('Pipeline exit code:', result.returncode)

## 3. Latest model predictions

Read the latest available prediction output. These are model-generated research signals, not executed brokerage orders.

In [ ]:
predictions_path = (
    PROJECT_ROOT / 'results/marketpulse_latest_predictions.csv'
)

predictions = pd.read_csv(predictions_path)
print('Prediction date:', predictions['date'].max())
print('Stocks evaluated:', len(predictions))
print('Selected signals:', int(predictions['signal'].sum()))
display(predictions)

## 4. Latest candidate trades

Display stocks meeting the saved probability threshold. Candidates are not automatically purchased.

In [ ]:
candidates = predictions.loc[
    predictions['signal'] == 1
].sort_values(
    'predicted_probability', ascending=False
)

display(candidates)

## 5. Historical backtest results

Review executed simulated trades and the momentum strategy equity curve.

In [ ]:
rf_trades = pd.read_parquet(
    PROJECT_ROOT / 'results/marketpulse_executed_trades.parquet'
)
momentum_equity = pd.read_parquet(
    PROJECT_ROOT / 'results/momentum_daily_equity.parquet'
)

print('RF simulated trades:', len(rf_trades))
print('Momentum equity observations:', len(momentum_equity))
display(rf_trades.tail(10))
display(momentum_equity.tail(10))